# hbv-torch demo: calibrating a differentiable HBV model on a CAMELS basin

This notebook loads one CAMELS-US basin, runs HBV with default parameters, calibrates it by gradient descent, and checks the water balance.

In [ ]:
import torch, matplotlib.pyplot as plt
from hbv_torch import HBV, nse, kge
from hbv_torch.data import load_camels_basin
from hbv_torch.calibrate import fit, to_tensor

CAMELS_ROOT = 'data/CAMELS'   # folder with basin_mean_forcing/ and usgs_streamflow/
BASIN = '01022500'

## 1. Load the basin
Forcing from Daymet, streamflow converted from cfs to mm/day, PET from Hargreaves.

In [ ]:
df = load_camels_basin(CAMELS_ROOT, BASIN)
print(df.attrs)
df.loc['2000':'2001', ['precip', 'qobs']].plot(subplots=True, figsize=(10, 4));

## 2. Calibrate
Water year 1999 is warm-up; 2000–2008 is calibration.

In [ ]:
cal = df.loc['1998-10-01':'2008-09-30']
warmup = len(df.loc['1998-10-01':'1999-10-01']) - 1
model = HBV(n_basins=1)
history = fit(model, to_tensor(cal.precip), to_tensor(cal.tmean), to_tensor(cal.pet), to_tensor(cal.qobs),
              warmup=warmup, epochs=300, lr=0.05)
model.named_params()

## 3. Validate and check the water balance

In [ ]:
full = df.loc['1998-10-01':'2014-09-30']
with torch.no_grad():
    out = model(to_tensor(full.precip, torch.float64), to_tensor(full.tmean, torch.float64),
                to_tensor(full.pet, torch.float64), params=model.params.double())
sim = out.q_routed[0].numpy()
val = full.index >= '2008-10-01'
print('validation NSE', float(nse(torch.tensor(full.qobs.values[val]), torch.tensor(sim[val]))))
print('validation KGE', float(kge(torch.tensor(full.qobs.values[val]), torch.tensor(sim[val]))))
print('water-balance residual (mm):', float(out.water_balance_residual()[0]))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(full.index, full.qobs, 'k', lw=0.7, label='observed')
ax.plot(full.index, sim, lw=0.7, label='HBV')
ax.set_ylabel('mm/day'); ax.legend();

## 4. Where the water goes
Storages over time show how the model represents the watershed's memory.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
for ax, (name, x) in zip(axes, [('snow', out.snowpack), ('soil', out.soil_moisture), ('upper zone', out.suz), ('lower zone', out.slz)]):
    ax.plot(full.index, x[0]); ax.set_ylabel(name + ' (mm)')